## Load Profile Assignment

Map each configured critical facility to a ResStock or ComStock archetype keyed by the configured ASHRAE climate zone and write the protocol-locked `load_profile_assignments.parquet` artifact.

Load profile assignment uses the DOE/NLR End-Use Load Profiles for the U.S. Building Stock dataset as the source for modeled facility load shapes. Each configured critical facility is matched to the closest ResStock or ComStock archetype using facility function, building type, and the configured ASHRAE/IECC climate zone; the selected archetype, climate-zone key, scaling basis, and schedule overlay are then written to the protocol-locked load_profile_assignments.parquet artifact for reproducibility.

Schedule overlays by facility type:
- `24x7` — police, fire, EOC, communications
- `business_hours` — town hall, library, senior center, post office
- `school_calendar` — K-12 schools
- `extended_residential` — public housing
- `industrial_continuous` — wastewater treatment

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from study_location import bootstrap
session = bootstrap()
location_root = session.location_root
repo_root = session.repo_root

# load Location Workspace config and standard grid paths.
from location_runtime import build_grid_paths
from location_runtime import load_runtime as load_config_runtime

config, paths = load_config_runtime(location_root / "config.yaml")
grid = build_grid_paths(config)
location_root = paths["location_root"]
location_name = paths["location_name"]
repo_root = paths["repo_root"]
for key in (
    "shift_cache",
    "opendss_root",
    "asset_registry",
    "augmented_artifacts",
    "onm_export",
    "figures",
):
    grid[key].mkdir(parents=True, exist_ok=True)

augmented_artifacts: /home/grahamhults/projects/Flood-RM/locations/marshfield/data/static/power_grid/smart_ds_compat


Read the critical-facility and critical-load assignment artifacts written by `01_der_inventory.ipynb`.

In [2]:
critical_facilities = pd.read_parquet(grid["augmented_artifacts"] / "critical_facilities.parquet")
load_matches = pd.read_parquet(grid["augmented_artifacts"] / "critical_load_assignments.parquet")

load_match_by_facility = {
    row["facility_id"]: row
    for row in load_matches.to_dict("records")
    if row.get("assignment_status") == "assigned"
}
print(f"{len(critical_facilities):,} critical facilities, {len(load_match_by_facility):,} assigned load profiles")

16 critical facilities, 16 assigned load profiles


### Assign Load Profile Archetypes

Assign each critical facility to a ResStock/ComStock archetype and write the protocol artifact with schedule-overlay provenance.

In [3]:
# build critical load, DER, switch, and block artifacts.
from power.resilience import load_profile_assignments_schema_version, load_inputs

load_profile_assignments_path = grid["augmented_artifacts"] / "load_profile_assignments.parquet"
oedi_profile_cache_dir = grid["augmented_artifacts"] / "oedi_eulp_cache"
use_oedi_load_profiles = False

profile_inputs = load_inputs(
    critical_facilities.to_dict("records"),
    load_match_by_facility,
    load_profile_assignments_path=load_profile_assignments_path,
    oedi_profile_cache_dir=oedi_profile_cache_dir,
    use_oedi_load_profiles=use_oedi_load_profiles,
)
assignments = pd.DataFrame(profile_inputs.assignment_rows)
display(assignments.head())
schedule_overlays = pd.Series(profile_inputs.load_profile_provenance_by_facility).map(lambda value: value.get("schedule_overlay"))
print("\nSchedule overlay distribution:")
display(schedule_overlays.value_counts())

,sandbox_id,load_asset_id,loadshape_id,municipality_id,tile_id,feeder_id,customer_class,profile_source,profile_source_version,source_geography,...,npts,p_scale_factor,q_scale_factor,annual_energy_kwh,peak_kw,power_factor_policy,diversity_group_id,rng_seed,source_provenance,schema_version
0,marshfield,marshfield:asset:load_buses:marshfield_shift_s...,marshfield:loadshape:town_hall_870_moraine_str...,marshfield:municipality:marshfield,None,None,critical_facility,comstock,synthetic_overlay_v0,ASHRAE_5A,...,8760,1.0,0.0,146080.602468,50.0,static_load_pf,town_hall_870_moraine_street_1525347a,0,"{""criticality_tier"": ""tier_1_response"", ""facil...",stage_b_load_profile_assignments.v0.1
1,marshfield,marshfield:asset:load_buses:marshfield_shift_s...,marshfield:loadshape:police_eoc_1639_ocean_str...,marshfield:municipality:marshfield,None,None,critical_facility,comstock,synthetic_overlay_v0,ASHRAE_5A,...,8760,1.0,0.0,607703.095620,75.0,static_load_pf,police_eoc_1639_ocean_street_414d3d42,0,"{""criticality_tier"": ""tier_1_response"", ""facil...",stage_b_load_profile_assignments.v0.1
2,marshfield,marshfield:asset:load_buses:marshfield_shift_s...,marshfield:loadshape:central_fire_station_60_s...,marshfield:municipality:marshfield,None,None,critical_facility,comstock,synthetic_overlay_v0,ASHRAE_5A,...,8760,1.0,0.0,486162.476350,60.0,static_load_pf,central_fire_station_60_south_river_street_ce1...,0,"{""criticality_tier"": ""tier_1_response"", ""facil...",stage_b_load_profile_assignments.v0.1
3,marshfield,marshfield:asset:load_buses:marshfield_shift_s...,marshfield:loadshape:dpw_building_965_plain_st...,marshfield:municipality:marshfield,None,None,critical_facility,comstock,synthetic_overlay_v0,ASHRAE_5A,...,8760,1.0,0.0,233728.964262,80.0,static_load_pf,dpw_building_965_plain_street_88deb37f,0,"{""criticality_tier"": ""tier_2_lifeline_support""...",stage_b_load_profile_assignments.v0.1
4,marshfield,marshfield:asset:load_buses:marshfield_shift_s...,marshfield:loadshape:governor_winslow_school_6...,marshfield:municipality:marshfield,None,None,critical_facility,comstock,synthetic_overlay_v0,ASHRAE_5A,...,8760,1.0,0.0,349298.631040,200.0,static_load_pf,governor_winslow_school_60_regis_road_4d66ca1c,0,"{""criticality_tier"": ""tier_2_lifeline_support""...",stage_b_load_profile_assignments.v0.1



Schedule overlay distribution:


school_calendar 7
24x7 4
business_hours 3
industrial_continuous 1
extended_residential 1
Name: count, dtype: int64

### Write Artifact

Write the protocol-locked `load_profile_assignments.parquet`

In [4]:
print(f"Wrote {len(assignments):,} rows to {load_profile_assignments_path.relative_to(repo_root)}")
print(f"Schema version: {load_profile_assignments_schema_version}")

Wrote 16 rows to locations/marshfield/data/static/power_grid/smart_ds_compat/load_profile_assignments.parquet
Schema version: stage_b_load_profile_assignments.v0.1
